<a href="https://colab.research.google.com/github/mdmostafizurrahman6351/Deep_Learning_Project/blob/main/Project_05_Cat_Vs_dog_classifier_using_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Cat vs dog model creating

import packge

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image
import os
import zipfile
from PIL import Image
import pathlib
from matplotlib.pyplot import imshow
import matplotlib.pyplot as plt

data load from microsoft dataset

In [ ]:
!wget https://download.microsoft.com/download/3/e/1/3e1c3f21-ecdb-4869-8368-6deba77b919f/kagglecatsanddogs_5340.zip

data preprocessing and splitting

In [ ]:
# data unzip
zip_ref = zipfile.ZipFile('/content/kagglecatsanddogs_5340.zip', 'r')
zip_ref.extractall()
zip_ref.close()


In [ ]:
# remove corrept image

def removed_correpted_images(path):
  count = 0
  for folder in ['Cat', 'Dog']:
    folder_path = os.path.join(path, folder)
    for file in os.listdir(folder_path):
      file_path = os.path.join(folder_path, file)
      try:
        img = Image.open(file_path).convert('RGB')
        img.save(file_path)
        img.verify()

      except:
        os.remove(file_path)
        count += 1
  print(f'{count} images removed')

removed_correpted_images('/content/PetImages')


In [ ]:
# train and validation data separate
data_dir = pathlib.Path('/content/PetImages')

train_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  validation_split=0.2,
  subset='training',
  seed=42,
  image_size=(150,150),
  batch_size=32
)

val_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  validation_split=0.2,
  subset='validation',
  seed=42,
  image_size=(150,150),
  batch_size=32
)

class_names = train_ds.class_names
print('Class name: ', class_names)

In [ ]:
# normalaizetion and perfornance
train_ds = train_ds.map(lambda x, y: (x/255, y))
val_ds = val_ds.map(lambda x, y: (x/255, y))

autotune = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=autotune)
val_ds = val_ds.cache().prefetch(buffer_size=autotune)
print('data loaded')

In [ ]:
# data augmantation

data_augmatation = tf.keras.Sequential([
  layers.RandomFlip("horizontal"),
  layers.RandomRotation(0.1),
  layers.RandomZoom(0.1)
])
print('data augmantation loaded')

Build CNN model

In [ ]:
# model create
model = models.Sequential([
  data_augmatation,
  layers.Conv2D(32, (3,3), activation='relu', input_shape=(150,150,3)),
  layers.MaxPooling2D((2,2)),

  layers.Conv2D(64, (3,3), activation='relu'),
  layers.MaxPooling2D((2,2)),

  layers.Conv2D(128, (3,3), activation='relu'),
  layers.MaxPooling2D((2,2)),

  layers.Flatten(),
  layers.Dense(512, activation='relu'),
  layers.Dense(1, activation='sigmoid')
])
print('model created')

In [ ]:
# model compile
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
print('model compiled')

In [ ]:
# model fit
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

model evulate

In [ ]:
loss, accuracy = model.evaluate(val_ds)
print(f'Loss: {loss}, Accuracy: {accuracy}')

predicted system

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Assuming model and val_ds are available from previous cells.

y_true_labels = []
y_pred_probs = []

# Collect true labels and predictions
for images, labels in val_ds:
    y_true_labels.extend(labels.numpy())
    y_pred_probs.extend(model.predict(images).flatten())

# Convert probabilities to binary predictions (0 or 1) using a 0.5 threshold
y_pred = (np.array(y_pred_probs) > 0.5).astype(int)
y_true = np.array(y_true_labels)

confu_matrix = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(confu_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Cat', 'Dog'], yticklabels=['Cat', 'Dog'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# image path
img_path = input('Enter image path: ')
img = Image.open(img_path)

# image view
plt.imshow(img)
plt.axis('off')
plt.show()

# image preprocessing
img = img.convert('RGB')
resized_img = img.resize((150,150))
img_array = tf.keras.utils.img_to_array(resized_img)
img_array = tf.expand_dims(img_array, 0)

# prediction
prediction = model.predict(img_array)
score = prediction[0][0]

# result
prediction_class = ''
confidance = 0
if score > 0.5:
  prediction_class = 'Dog'
  confidance = score * 100
else:
  prediction_class = 'Cat'
  confidance = (1 - score) * 100

print('Prediction Class: ', prediction_class)
print('Confidance: ', confidance, '%')